In [1]:
import os
import polars as pl
from tqdm.notebook import tqdm

pl.Config(tbl_rows=50)

In [2]:
data_path = '../data/meds_outliers/'

In [3]:
data = pl.read_parquet('../data/meds_outliers/data/train/1.parquet')

In [4]:
# pl.Config(fmt_str_lengths=100000)
# data.with_columns(pl.col('code').str.split('//').list.len().alias('length')).group_by(['code_type','length']).first()['code_type','length','code']

In [5]:
data.filter(pl.col('code_type') == 'ICU-FLUID-OUTPUT').head(1)

subject_id,seq_id,out_id,er_id,hadm_id,icustay_id,disch_id,time,code,numeric_value,text_value,itemid,died_in_hosp,icu_los,admission_type,admission_location,discharge_location,diag_version,diag_icd_code,diag_seq_num,drg_severity,drg_mortality,drg_type,drg_code,priority,specimen_id,lab_lower_limit,lab_upper_limit,lab_flag,lab_unit,lab_itemid,gender,route,frequency,doses_per_24_hrs,medication,proc_seq_num,proc_version,proc_icd_code,micro_specimen_id,micro_org_name,micro_test_name,micro_spec_type_desc,micro_test_itemid,icu_care_unit,category,label,abbreviation,rate,unit,amount,amountuom,ordercategorydescription,ordercategoryname,secondaryordercategoryname,ordercomponenttypedescription,table,race,code_type,icd9_to_icd10_d,icd9_to_icd10_p,clean_medication,lab_label,lab_fluid,lab_category,lab_description,lab_frequency,time_diff,numeric_value/is_inlier
i64,f64,f64,f64,f64,f64,f64,datetime[μs],str,f64,str,f64,f64,f64,str,str,str,f64,str,f64,f64,f64,str,f64,str,f64,f64,f64,str,str,f64,str,str,str,f64,str,f64,f64,str,f64,str,str,str,f64,str,str,str,str,f64,str,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,f64,bool
10052769,2.2087051e7,null,null,2.2087051e7,3.883265e7,null,2124-04-26 13:41:00,"""ICU-FLUID-OUTPUT//226560//Void""",400.0,null,226560.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""Output""","""Void""","""Void""",null,null,null,null,null,null,null,null,"""icu/inputevents""",null,"""ICU-FLUID-OUTPUT""",null,null,"""UNK""",null,null,null,null,null,0.007639,true


In [6]:
descemb_mapping = {'MICROBIOLOGY': ['micro_test_name'],
'DIAGNOSIS-ICD': ['code_type','icd9_to_icd10_d'],
'MEDS_DEATH': ['code_type'],
'EMERGENCY-END': ['code_type'],
'EMERGENCY-START': ['code_type'],
'ADMISSION-AT-HOSPITAL': ['code_type'],
'MEDICATION': ['clean_medication','route','doses_per_24_hrs'],
'DISCHARGE-FROM-ICU': ['code_type'],
'OUTPATIENT-END': ['code_type'],
'ICU-CHART': ['label','numeric_value' 'unitname'],
'DISCHARGE-FROM-HOSPITAL': ['code_type'],
'ADMISSION-LOCATION': ['code_type'], #  and use .split('//')[-1] as second element of the list]
'LAB': ['lab_label', 'numeric_value', 'lab_unit'],
'RACE': ['code_type','race'],
'AGE_AT_ADMISSION': ['code_type','numeric_value'],
'PROCEDURE-ICD': ['code_type','icd9_to_icd10_p'],
'TIME-GAP': ['code_type','numeric_value'],
'DRG': ['code_type','drg_type','drg_code', 'drg_mortality'],
'ADMISSION-TYPE':['code_type'], #  and use .split('//')[-1] as second element of the list]
'ADMISSION-AT-ICU': ['code_type'],
'ICU-INFUSION': ['label','numeric_value', 'amountuom'],
'DISCHARGE-lOCATION': ['code_type','discharge_location'],
'GENDER': ['code_type','gender'],
'ICU-FLUID-OUTPUT': ['label','numeric_value' 'unitname'],
'OUTPATIENT-START': ['code_type'],
'ICU-PROCEDURE': ['label'],
}

In [7]:
import polars as pl

def dsva_number(x):
    """
    Digit-Split Value Aggregation with max 3 decimals:
    12.34567  -> '1 2 . 3 5'
    5         -> '5'
    5.1       -> '5 . 1'
    """
    if x is None:
        return None
    try:
        v = float(x)
    except (TypeError, ValueError):
        return None

    # round to 3 decimal places
    v = round(v, 3)

    # format to 3 decimals, then strip trailing zeros and dot
    s = f"{v:.3f}".rstrip("0").rstrip(".")

    # DSVA: split into individual characters separated by spaces
    return " ".join(list(s))


def dsva_expr(col: pl.Expr) -> pl.Expr:
    return (
        col.cast(pl.Float64)
           .map_elements(dsva_number, return_dtype=pl.Utf8)
    )


In [8]:
gender_norm = (
    pl.when(pl.col("gender").str.to_lowercase() == "m")
      .then(pl.lit("male"))
    .when(pl.col("gender").str.to_lowercase() == "f")
      .then(pl.lit("female"))
    .otherwise(pl.col("gender"))
)


In [9]:
def normalize_code_type():
    return (
        pl.col("code_type")
        .str.replace_all(r"[_\-]+", " ")   # replace underscores and dashes with space
        .str.to_lowercase()
        .str.strip()
    )

In [10]:
def add_descemb(df: pl.DataFrame) -> pl.DataFrame:
    ct = pl.col("code_type")
    code = pl.col("code")

    # clean code type: remove - and _, lowercase
    code_type_clean = (
        ct.str.replace_all(r"[_\-]+", " ")
          .str.to_lowercase()
          .str.strip_chars()
    )

    # last element after the last "//"
    code_tail = code.str.extract(r"([^/]+)$").str.to_lowercase()

    descemb_expr = (
        pl.when(ct == "MICROBIOLOGY")
          .then(
              pl.col("micro_test_name").str.to_lowercase()
          )

        .when(ct == "DIAGNOSIS-ICD")
          .then(
              pl.concat_str(
                  [pl.lit("diagnosis icd 10 code"), pl.col("icd9_to_icd10_d")],
                  separator=" ",
                  ignore_nulls=True
              )
          )

        .when(ct == "MEDS_DEATH")
          .then(code_type_clean)

        .when(ct == "EMERGENCY-END")
          .then(code_type_clean)

        .when(ct == "EMERGENCY-START")
          .then(code_type_clean)

        .when(ct == "ADMISSION-AT-HOSPITAL")
          .then(code_type_clean)

        .when(ct == "MEDICATION")
          .then(
              pl.concat_str(
                  [
                      pl.col("clean_medication"),
                      pl.col("route"),
                      pl.col("doses_per_24_hrs")
                  ],
                  separator=" ",
                  ignore_nulls=True
              ).str.to_lowercase()
          )

        .when(ct == "DISCHARGE-FROM-ICU")
          .then(code_type_clean)

        .when(ct == "OUTPATIENT-END")
          .then(code_type_clean)

        .when(ct == "ICU-CHART")
          .then(
              pl.concat_str(
                  [
                      pl.col("label"),
                      dsva_expr(pl.col("numeric_value")),
                      pl.col("unitname")  # change if your col is named differently
                  ],
                  separator=" ",
                  ignore_nulls=True
              ).str.to_lowercase()
          )

        .when(ct == "DISCHARGE-FROM-HOSPITAL")
          .then(code_type_clean)

        .when(ct == "ADMISSION-LOCATION")
          .then(
              pl.concat_str(
                  [code_type_clean, code_tail],
                  separator=" ",
                  ignore_nulls=True
              )
          )

        .when(ct == "LAB")
          .then(
              pl.concat_str(
                  [
                      pl.col("lab_label"),
                      dsva_expr(pl.col("numeric_value")),
                      pl.col("lab_unit")
                  ],
                  separator=" ",
                  ignore_nulls=True
              ).str.to_lowercase()
          )

        .when(ct == "RACE")
          .then(
              pl.concat_str(
                  [code_type_clean, pl.col("race")],
                  separator=" ",
                  ignore_nulls=True
              ).str.to_lowercase()
          )

        .when(ct == "AGE_AT_ADMISSION")
          .then(
              pl.concat_str(
                  [
                      code_type_clean,
                      dsva_expr(pl.col("numeric_value"))
                  ],
                  separator=" ",
                  ignore_nulls=True
              )
          )

        .when(ct == "PROCEDURE-ICD")
          .then(
              pl.concat_str(
                  [pl.lit("procedure icd 10 code"), pl.col("icd9_to_icd10_p")],
                  separator=" ",
                  ignore_nulls=True
              ).str.to_lowercase()
          )


        .when(ct == "TIME-GAP")
          .then(
              pl.concat_str(
                  [code_type_clean, dsva_expr(pl.col("numeric_value"))],
                  separator=" ",
                  ignore_nulls=True
              )
          )

        .when(ct == "DRG")
          .then(
              pl.concat_str(
                  [   code_type_clean,
                      pl.col("drg_type"),
                      pl.col("drg_code"),
                      pl.col("drg_mortality")
                  ],
                  separator=" ",
                  ignore_nulls=True
              ).str.to_lowercase()
          )

        .when(ct == "ADMISSION-TYPE")
          .then(
              pl.concat_str(
                  [code_type_clean, code_tail],
                  separator=" ",
                  ignore_nulls=True
              )
          )

        .when(ct == "ADMISSION-AT-ICU")
          .then(code_type_clean)

        .when(ct == "ICU-INFUSION")
          .then(
              pl.concat_str(
                  [
                      pl.col("label"),
                      dsva_expr(pl.col("numeric_value")),
                      pl.col("amountuom")
                  ],
                  separator=" ",
                  ignore_nulls=True
              ).str.to_lowercase()
          )

        .when(ct == "DISCHARGE-lOCATION")
          .then(
              pl.concat_str(
                  [code_type_clean, pl.col("discharge_location")],
                  separator=" ",
                  ignore_nulls=True
              ).str.to_lowercase()
          )

        .when(ct == "GENDER")
          .then(
              pl.concat_str(
                  [code_type_clean, gender_norm],
                  separator=" ",
                  ignore_nulls=True
              )
          )

        .when(ct == "ICU-FLUID-OUTPUT")
          .then(
              pl.concat_str(
                  [
                      pl.col("label"),
                      dsva_expr(pl.col("numeric_value")),
                      pl.col("unitname")  # change if needed
                  ],
                  separator=" ",
                  ignore_nulls=True
              ).str.to_lowercase()
          )

        .when(ct == "OUTPATIENT-START")
          .then(code_type_clean)

        .when(ct == "ICU-PROCEDURE")
          .then(
              pl.col("label").str.to_lowercase()
          )

        .otherwise(pl.lit(None))
        .alias("descemb")
    )



    return df.with_columns(descemb_expr)#['subject_id',
#                                         'seq_id',
#                                         'out_id',
#                                         'er_id',
#                                         'hadm_id',
#                                         'icustay_id',
#                                         'disch_id',
#                                         'time',
#                                         'code',
#                                         'code_type',
#                                         'descemb']


In [11]:
genhpf_mapping = {'MICROBIOLOGY': ['code_type','micro_test_name', 'micro_spec_type_desc'],
'DIAGNOSIS-ICD': ['code_type','icd9_to_icd10_d'],
'MEDS_DEATH': ['code_type'],
'EMERGENCY-END': ['code_type'],
'EMERGENCY-START': ['code_type'],
'ADMISSION-AT-HOSPITAL': ['code_type'],
'MEDICATION': ['code_type','clean_medication','route','frequency','doses_per_24_hrs'],
'DISCHARGE-FROM-ICU': ['code_type'],
'OUTPATIENT-END': ['code_type'],
'ICU-CHART': ['code_type','category','label','numeric_value', 'unitname'],
'DISCHARGE-FROM-HOSPITAL': ['code_type'],
'ADMISSION-LOCATION': ['code_type'], #  and use .split('//')[-1] as second element of the list]
'LAB': ['code_type','lab_label', 'priority' ,'numeric_value', 'lab_unit','lab_lower_limit', 'lab_upper_limit', 'lab_flag'],
'RACE': ['code_type','race'],
'AGE_AT_ADMISSION': ['code_type','numeric_value'],
'PROCEDURE-ICD': ['code_type','icd9_to_icd10_p'],
'TIME-GAP': ['code_type','numeric_value'],
'DRG': ['code_type','drg_type','drg_code', 'drg_mortality'],
'ADMISSION-TYPE':['code_type'], #  and use .split('//')[-1] as second element of the list]
'ADMISSION-AT-ICU': ['code_type'],
'ICU-INFUSION': ['code_type', 'category', 'label','numeric_value', 'amountuom'],
'DISCHARGE-lOCATION': ['code_type','discharge_location'],
'GENDER': ['code_type','gender'],
'ICU-FLUID-OUTPUT': ['code_type','category','label','numeric_value', 'unitname'],
'OUTPATIENT-START': ['code_type'],
'ICU-PROCEDURE': ['code_type','label'],
}

In [12]:
FEATURE_NAME_MAP = {
    "micro_test_name": "test name",
    "micro_spec_type_desc": "test description",
    "clean_medication": "medication name",
    "doses_per_24_hrs": "dose per 24 hours",
    "numeric_value": "value",
    "lab_label": "lab name",
    "lab_unit": "unit",
    "lab_flag": "flag",
    "amountuom": "unit",
    "unitname": "unit",
    "icd9_to_icd10_d": "value",
    "icd9_to_icd10_p": "value",
}

In [13]:
CODETYPE = (
    pl.col("code_type")
      .str.replace_all(r"[_\-]+", " ")
      .str.to_lowercase()
      .str.strip_chars()
)

In [14]:
def value_expr(col: pl.Expr, name: str) -> pl.Expr:
    if name in ["numeric_value"]:
        return dsva_expr(col)
    if name in ["lab_lower_limit", "lab_upper_limit"]:
        return dsva_expr(col)
    if name in ["icd9_to_icd10_d", "icd9_to_icd10_p"]:
        return col  # already textual ICD10 code
    return col  # textual columns

In [15]:
def feature_pair(feature_col: str) -> pl.Expr:
    """Return expression: '<feature name> <value>' """
    feat_name = FEATURE_NAME_MAP.get(feature_col, feature_col.replace("_", " ").replace("-", " "))
    feat_name = feat_name.lower()

    col_expr = pl.col(feature_col)
    col_expr = value_expr(col_expr, feature_col)

    return pl.concat_str([pl.lit(f"{feat_name}: "), col_expr], separator="", ignore_nulls=True)


In [17]:
def build_genhpf_desc(df: pl.DataFrame, mapping: dict) -> pl.DataFrame:
    ct_clean = CODETYPE
    code_tail = pl.col("code").str.extract(r"([^/]+)$").str.to_lowercase()

    # Build master expression
    expr = pl.lit("")  # will be overridden per code_type

    for code_type, cols in mapping.items():
        parts = []

        # always start with cleaned code type:
        parts.append(
                    pl.concat_str(
                        [pl.lit("event type: "), ct_clean],
                        separator="",
                        ignore_nulls=True
                    )
                )

        # add each feature pair
        for col in cols:
            if col == "code_type":
                continue
            parts.append(feature_pair(col))

        # build text for this code_type
        text_expr = (
            pl.concat_str(parts, separator=", ", ignore_nulls=True)
              .str.to_lowercase()
        )

        # add conditional branch
        expr = (
            pl.when(pl.col("code_type") == code_type)
              .then(text_expr)
              .otherwise(expr)
        )

    return df.with_columns(expr.alias("genhpf"))['subject_id',
                                        'seq_id',
                                        'out_id',
                                        'er_id',
                                        'hadm_id',
                                        'icustay_id',
                                        'disch_id',
                                        'code',
                                        'time',
                                        'descemb',
                                        'genhpf']


In [19]:
data_path = '../data/meds_outliers/data/train/'
out_path = '../data/descemb_genhpf/data/train/'
d_item = pl.read_csv('../resources/mimic-mapping/d_items.csv',
                     infer_schema_length=100000,
                    columns=['itemid','unitname'])

for shard in tqdm(os.listdir(data_path)):
    data = pl.read_parquet(os.path.join(data_path,shard))
    data = data.with_columns(pl.col("itemid").cast(pl.Int64))
    data = data.join(d_item, on='itemid', how='left')
    data = add_descemb(data)
    data = build_genhpf_desc(data, genhpf_mapping)
    data.write_parquet(os.path.join(out_path,shard))

  0%|          | 0/365 [00:00<?, ?it/s]